# NeuroLens — Precompute Cache

Run this notebook **once** on a GPU runtime to generate the NeuroLens cache.
This processes the stimulus library through TRIBE v2 and comparison models.

**Requirements:** Colab GPU runtime, HuggingFace account (for LLaMA access)

In [ ]:
# --- Kaggle HuggingFace Auth (uncomment if running on Kaggle) ---
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login
# import os
#
# user_secrets = UserSecretsClient()
# try:
#     hf_token = user_secrets.get_secret("HF_TOKEN")
#     print(f"Token retrieved successfully. Length: {len(hf_token)}")
# except Exception as e:
#     print(f"Error retrieving secret: {e}")
#     hf_token = None
#
# if hf_token:
#     login(token=hf_token)
#     os.environ["HF_TOKEN"] = hf_token
#     print("Logged in to Hugging Face.")
# else:
#     print("Token not found. Check Secrets label.")

import sys, os
from pathlib import Path

# Clone repo and set up environment (Colab)
if not os.path.exists('neurolens'):
    if not os.path.exists('tribev2'):
        !git clone https://github.com/Ansumanbhujabal/tribev2.git
    os.chdir('tribev2')

sys.path.insert(0, os.getcwd())

# Install all deps
!pip install -q numpy scipy av ffmpeg-python \
    exca neuralset neuraltrain einops pyyaml huggingface_hub \
    gtts langdetect spacy soundfile Levenshtein julius transformers x_transformers \
    nilearn matplotlib pydantic tqdm open_clip_torch 2>/dev/null

# Install tribev2 editable
!pip install -q -e '.[plotting]' --no-deps 2>/dev/null || true

# Install moviepy >= 2.1 (neuralset uses v2 API like .subclipped())
# then patch the import so 'from moviepy import VideoFileClip' works
!pip install -q 'moviepy>=2.1' --force-reinstall 2>/dev/null

import moviepy
from moviepy import VideoClip
# In moviepy v2, VideoFileClip moved to moviepy.video.io.VideoFileClip
# but neuralset does 'from moviepy import VideoFileClip'
from moviepy.video.io.VideoFileClip import VideoFileClip
moviepy.VideoFileClip = VideoFileClip
sys.modules['moviepy'].VideoFileClip = VideoFileClip
print(f'moviepy {moviepy.__version__} patched: VideoFileClip accessible from top-level')

import json
import numpy as np
import torch
from tribev2 import TribeModel

CACHE_DIR = Path('neurolens_cache')
CACHE_DIR.mkdir(exist_ok=True)
for subdir in ['stimuli', 'brain_preds', 'roi_summaries', 'embeddings']:
    (CACHE_DIR / subdir).mkdir(exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# Stimulus library
STIMULI = [
    {'id': 'clip_001', 'name': 'Jack Ma Motivation', 'category': 'Speech',
     'media_type': 'video', 'duration_sec': 10.0, 'path': 'neurolens/videos/Jack_ma_motivation.mp4'},
    {'id': 'clip_002', 'name': 'Muay Thai Kick', 'category': 'Silence + Visuals',
     'media_type': 'video', 'duration_sec': 8.0, 'path': 'neurolens/videos/Muay_thai_kick.mp4'},
    {'id': 'clip_003', 'name': 'Ronaldo Motivation', 'category': 'Speech',
     'media_type': 'video', 'duration_sec': 10.0, 'path': 'neurolens/videos/ronaldo_motivation.mp4'},
    {'id': 'clip_004', 'name': 'Romantic Couple', 'category': 'Silence + Visuals',
     'media_type': 'video', 'duration_sec': 15.0, 'path': 'neurolens/videos/romantic_couple.mp4'},
    {'id': 'clip_005', 'name': 'Food Reel', 'category': 'Silence + Visuals',
     'media_type': 'video', 'duration_sec': 12.0, 'path': 'neurolens/videos/food_reel.mp4'},
    {'id': 'clip_006', 'name': 'High Impact Music', 'category': 'Music',
     'media_type': 'video', 'duration_sec': 15.0, 'path': 'neurolens/videos/high_impact_music.mp4'},
]

# Save metadata (without paths)
metadata = [{k: v for k, v in s.items() if k != 'path'} for s in STIMULI]
(CACHE_DIR / 'stimuli' / 'metadata.json').write_text(json.dumps(metadata, indent=2))
print(f'Saved metadata for {len(STIMULI)} stimuli')

In [ ]:
# Load TRIBE v2 (run once, stays in memory)
import gc
from neurolens.roi import summarize_by_roi_group

model = TribeModel.from_pretrained("facebook/tribev2", cache_folder="./cache")
print("Model loaded.")

In [ ]:
# --- clip_001: Jack Ma Motivation ---
stim = STIMULI[0]
events = model.get_events_dataframe(video_path=stim["path"])
preds, segments = model.predict(events)
np.savez(CACHE_DIR / "brain_preds" / f"{stim['id']}.npz", preds=preds)
roi_summary = summarize_by_roi_group(preds.mean(axis=0))
(CACHE_DIR / "roi_summaries" / f"{stim['id']}.json").write_text(json.dumps(roi_summary))
print(f"{stim['id']}: {preds.shape[0]} timesteps, {preds.shape[1]} vertices")
del preds, segments, events, roi_summary; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# --- clip_002: Muay Thai Kick ---
stim = STIMULI[1]
events = model.get_events_dataframe(video_path=stim["path"])
preds, segments = model.predict(events)
np.savez(CACHE_DIR / "brain_preds" / f"{stim['id']}.npz", preds=preds)
roi_summary = summarize_by_roi_group(preds.mean(axis=0))
(CACHE_DIR / "roi_summaries" / f"{stim['id']}.json").write_text(json.dumps(roi_summary))
print(f"{stim['id']}: {preds.shape[0]} timesteps, {preds.shape[1]} vertices")
del preds, segments, events, roi_summary; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# --- clip_003: Ronaldo Motivation ---
stim = STIMULI[2]
events = model.get_events_dataframe(video_path=stim["path"])
preds, segments = model.predict(events)
np.savez(CACHE_DIR / "brain_preds" / f"{stim['id']}.npz", preds=preds)
roi_summary = summarize_by_roi_group(preds.mean(axis=0))
(CACHE_DIR / "roi_summaries" / f"{stim['id']}.json").write_text(json.dumps(roi_summary))
print(f"{stim['id']}: {preds.shape[0]} timesteps, {preds.shape[1]} vertices")
del preds, segments, events, roi_summary; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# --- clip_004: Romantic Couple ---
stim = STIMULI[3]
events = model.get_events_dataframe(video_path=stim["path"])
preds, segments = model.predict(events)
np.savez(CACHE_DIR / "brain_preds" / f"{stim['id']}.npz", preds=preds)
roi_summary = summarize_by_roi_group(preds.mean(axis=0))
(CACHE_DIR / "roi_summaries" / f"{stim['id']}.json").write_text(json.dumps(roi_summary))
print(f"{stim['id']}: {preds.shape[0]} timesteps, {preds.shape[1]} vertices")
del preds, segments, events, roi_summary; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# --- clip_005: Food Reel ---
stim = STIMULI[4]
events = model.get_events_dataframe(video_path=stim["path"])
preds, segments = model.predict(events)
np.savez(CACHE_DIR / "brain_preds" / f"{stim['id']}.npz", preds=preds)
roi_summary = summarize_by_roi_group(preds.mean(axis=0))
(CACHE_DIR / "roi_summaries" / f"{stim['id']}.json").write_text(json.dumps(roi_summary))
print(f"{stim['id']}: {preds.shape[0]} timesteps, {preds.shape[1]} vertices")
del preds, segments, events, roi_summary; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# --- clip_006: High Impact Music ---
stim = STIMULI[5]
events = model.get_events_dataframe(video_path=stim["path"])
preds, segments = model.predict(events)
np.savez(CACHE_DIR / "brain_preds" / f"{stim['id']}.npz", preds=preds)
roi_summary = summarize_by_roi_group(preds.mean(axis=0))
(CACHE_DIR / "roi_summaries" / f"{stim['id']}.json").write_text(json.dumps(roi_summary))
print(f"{stim['id']}: {preds.shape[0]} timesteps, {preds.shape[1]} vertices")
del preds, segments, events, roi_summary; gc.collect(); torch.cuda.empty_cache()

In [ ]:
import open_clip
from PIL import Image
import av

clip_model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='laion2b_s34b_b79k'
)
clip_model = clip_model.to(device).eval()

(CACHE_DIR / 'embeddings' / 'clip').mkdir(exist_ok=True)

def get_middle_frame(video_path):
    """Extract middle frame from video using PyAV."""
    container = av.open(video_path)
    stream = container.streams.video[0]
    total = stream.frames
    target = total // 2 if total > 0 else 0
    for i, frame in enumerate(container.decode(video=0)):
        if i >= target:
            return frame.to_image()  # Returns PIL Image
    return frame.to_image()

for stim in STIMULI:
    sid = stim['id']
    if stim['media_type'] != 'video':
        continue
    try:
        img = get_middle_frame(stim['path'])
        img_tensor = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            emb = clip_model.encode_image(img_tensor).squeeze().cpu()
        torch.save(emb, CACHE_DIR / 'embeddings' / 'clip' / f'{sid}.pt')
        print(f'CLIP embedding for {sid}: {emb.shape}')
    except Exception as e:
        print(f'Skipped CLIP for {sid}: {e}')

In [ ]:
from transformers import WhisperModel, WhisperFeatureExtractor
import soundfile as sf
import subprocess

whisper = WhisperModel.from_pretrained('openai/whisper-base').to(device).eval()
whisper_fe = WhisperFeatureExtractor.from_pretrained('openai/whisper-base')

(CACHE_DIR / 'embeddings' / 'whisper').mkdir(exist_ok=True)

for stim in STIMULI:
    sid = stim['id']
    # Extract audio at 16kHz mono (Whisper requirement)
    audio_path = f'/tmp/{sid}_16k.wav'
    try:
        subprocess.run(['ffmpeg', '-y', '-i', stim['path'], '-ar', '16000', '-ac', '1',
                       audio_path], capture_output=True, check=True)
        audio, sr = sf.read(audio_path)
        inputs = whisper_fe(audio, sampling_rate=16000, return_tensors='pt').to(device)
        with torch.no_grad():
            out = whisper.encoder(**inputs)
            emb = out.last_hidden_state.mean(dim=1).squeeze().cpu()
        torch.save(emb, CACHE_DIR / 'embeddings' / 'whisper' / f'{sid}.pt')
        print(f'Whisper embedding for {sid}: {emb.shape}')
    except Exception as e:
        print(f'Skipped Whisper for {sid}: {e}')

In [ ]:
from transformers import GPT2Model, GPT2Tokenizer

gpt2 = GPT2Model.from_pretrained("gpt2").to(device).eval()
gpt2_tok = GPT2Tokenizer.from_pretrained("gpt2")

(CACHE_DIR / "embeddings" / "gpt2").mkdir(exist_ok=True)

for stim in STIMULI:
    sid = stim["id"]
    text = f"{stim['name']}. Category: {stim['category']}"
    inputs = gpt2_tok(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = gpt2(**inputs)
        emb = out.last_hidden_state.mean(dim=1).squeeze().cpu()
    torch.save(emb, CACHE_DIR / "embeddings" / "gpt2" / f"{sid}.pt")
    print(f"GPT-2 embedding for {sid}: {emb.shape}")

## Done!

Cache generated at `neurolens_cache/`. Download this folder or upload it to
Google Drive / HuggingFace Hub for use with the main `neurolens.ipynb` notebook.